[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/qwen3_5_4b_decision_email_triage.ipynb)

# Qwen3.5-4B decision model for support email triage (Unsloth)

This notebook turns **Qwen3.5-4B** (`unsloth/Qwen3.5-4B`) into a **decision model** for customer-support email triage, following Unsloth's [Train your own Decision Model](https://unsloth.ai/docs/basics/train-your-own-decision-model-with-unsloth) guide.

A decision model does not generate text. Unsloth puts the email and every question with its options into one prompt, and a small [Clef](https://huggingface.co/Cloudflare/clef)-style head scores each option. Each answer is a choice plus a calibrated probability, the same shape as Jev, Laya and Clef answers. This notebook asks two questions per email:

- `category` (`choice`, 8 options): `billing`, `technical_issue`, `account_access`, `refund_request`, `shipping_delivery`, `feature_request`, `cancellation`, `spam_or_other`
- `priority` (`score`, 3 levels): `0` low, `1` normal, `2` urgent

The data is a shared synthetic dataset in [`colab/data/support_email_triage`](https://github.com/donvito/notebooks/tree/main/colab/data/support_email_triage). It has 1,600 train and 400 test emails, generated with a fixed seed. [`gemma4_e4b_decision_email_triage.ipynb`](gemma4_e4b_decision_email_triage.ipynb) trains Gemma 4 E4B on the same files and split, so you can compare the two runs directly.

Settings follow the guide: **4-bit LoRA, rank 16, learning rate 2e-4, 2 epochs**. The new head trains at its default 1e-4.

### GPU and VRAM

| GPU | Batch × grad accum | Expected peak VRAM (estimate) |
| --- | --- | --- |
| RTX 3090 / 4090, L4, A10G (24GB) | 8 × 2 | ~12–16 GB |
| A100 40/80GB | 8 × 2 | same as above |
| T4 (16GB) | 2 × 8 | ~10–14 GB, slow (see below) |

These are estimates, not measurements: this notebook was written without GPU access. The 4-bit Qwen3.5-4B backbone is about 3 GB. Each row is 550–680 tokens, so `max_seq_length = 1024` never truncates and activations stay small. The guide's own Qwen3.5-4B run used batch 8 × 4 at `max_seq_length = 2048`. That is far more than this notebook needs, and it still fits a 24GB card. The cell after training prints the real peak.

**T4 caveat:** a T4 has no bfloat16. Qwen3.5's gated delta-net layers cannot train in float16, so Unsloth runs Qwen3.5 in float32 there. That uses more memory and is much slower. Use an L4 or A100 in Colab when you can.

In Colab, use **Runtime → Change runtime type → L4 GPU** (or A100). The same notebook runs unchanged on a local **RTX 3090 / 4090 (24GB)**.

In [ ]:
!nvidia-smi

## 1. Install Unsloth

Decision models (`FastDecisionModel`, `DecisionTrainer`) shipped in `unsloth` **2026.10.2**. The guide's install is `pip install --upgrade unsloth`. If Colab asks you to restart the session after the install, restart it and continue from the next cell.

In [ ]:
!pip install -q --upgrade "unsloth>=2026.10.2" "unsloth_zoo>=2026.10.2"
!pip install -q scikit-learn matplotlib pandas

## 2. Configuration

Batch size is picked from the GPU's memory, so the same cell stays under 24GB on a 3090 and still fits on smaller cards. Effective batch size is 16 either way, so both notebooks take the same number of optimizer steps.

In [ ]:
import torch

# ---------- Model ----------
MODEL_NAME = "unsloth/Qwen3.5-4B"
MAX_SEQ_LENGTH = 1024   # longest triage row is ~680 tokens with this tokenizer, so nothing is cut
LOAD_IN_4BIT = True

# ---------- LoRA + training (the guide's recommended settings) ----------
LORA_R = 16
LORA_ALPHA = 16
LEARNING_RATE = 2e-4
EPOCHS = 2
SEED = 3407

gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 0
if gpu_gb >= 22:      # RTX 3090 / 4090, L4, A10G, A100
    BATCH_SIZE, GRAD_ACCUM = 8, 2
else:                 # T4 (16GB) and other smaller cards
    BATCH_SIZE, GRAD_ACCUM = 2, 8

# ---------- Data ----------
# Notebooks load the committed JSONL files from this repo. To test a branch before it
# is merged, set DATA_REF to the branch name.
DATA_REPO = "donvito/notebooks"
DATA_REF = "main"
EXPECTED_SHA256 = {
    "train.jsonl": "7b1d86a2d217a93bc019e70ebab41bb3968cb9906eefb36e9a156a8f915a3553",
    "test.jsonl": "2bbadbbfb700a828083284acd0c90677b63c155ec81de83bab18175415c9e8f0",
}

# ---------- Outputs ----------
LORA_DIR = "qwen3_5-4b-email-triage-lora"
MERGED_DIR = "qwen3_5-4b-email-triage-merged"
SAVE_MERGED = True   # 16-bit merged copy, needed to serve the model from the Decision API

print(f"GPU memory: {gpu_gb:.1f} GB -> batch {BATCH_SIZE} x grad accum {GRAD_ACCUM}")

## 3. Load the shared triage dataset

Each row uses the guide's dataset format, the same layout as [`LocalLLaMA/typed-decisions`](https://huggingface.co/datasets/LocalLLaMA/typed-decisions). It has a `state` (the email), the `questions` to decide, and the `gold` answers. Each is stored as a JSON string. `gold` holds a `label` and `probabilities`. About 14% of emails are deliberately ambiguous: they mention a second topic, so their `category` gold is soft (for example 0.7 `cancellation` / 0.3 `refund_request`). The flat `category`, `priority` and `ambiguous` columns are only for reporting.

The cell uses the files next to this notebook when you run from a clone of the repo. Otherwise it downloads them from GitHub. It then checks their SHA-256, so both notebooks provably train and test on identical data. To rebuild the files, run `python colab/data/support_email_triage/generate.py` (standard library only, seed 3407).

In [ ]:
import hashlib, json, urllib.request
from pathlib import Path
import pandas as pd

def dataset_dir():
    for candidate in (Path("data/support_email_triage"), Path("colab/data/support_email_triage")):
        if all((candidate / name).is_file() for name in EXPECTED_SHA256):
            return candidate
    target = Path("support_email_triage")
    target.mkdir(exist_ok=True)
    for name in EXPECTED_SHA256:
        url = f"https://raw.githubusercontent.com/{DATA_REPO}/{DATA_REF}/colab/data/support_email_triage/{name}"
        urllib.request.urlretrieve(url, target / name)
    return target

DATA_DIR = dataset_dir()

def load_split(name):
    raw = (DATA_DIR / name).read_bytes().replace(b"\r\n", b"\n")
    digest = hashlib.sha256(raw).hexdigest()
    assert digest == EXPECTED_SHA256[name], f"{name} differs from the shared dataset: {digest}"
    return [json.loads(line) for line in raw.decode("utf-8").splitlines() if line.strip()]

train_rows = load_split("train.jsonl")
test_rows = load_split("test.jsonl")
print(f"Loaded {len(train_rows)} train / {len(test_rows)} test rows from {DATA_DIR} (SHA-256 verified)")

QUESTIONS = json.loads(train_rows[0]["questions"])
CATEGORIES = list(QUESTIONS["category"]["criteria"])
PRIORITY_LEVELS = QUESTIONS["priority"]["criteria"]

summary = pd.DataFrame(
    [{"split": r["split"], "category": r["category"], "priority": r["priority"], "ambiguous": r["ambiguous"]}
     for r in train_rows + test_rows]
)
display(pd.crosstab(summary["category"], summary["split"]))
display(pd.crosstab(summary["priority"], summary["split"]))

In [ ]:
example = train_rows[0]
email = json.loads(example["state"])
print("From:   ", email["from"])
print("Subject:", email["subject"])
print()
print(email["body"])
print()
print("Gold:", json.dumps(json.loads(example["gold"]), indent=2)[:600])

## 4. Load Qwen3.5-4B as a decision model

`FastDecisionModel.from_pretrained` loads the language model in 4-bit and attaches a fresh decision head. The head is random until it is trained, which is why the baseline below sits near chance. Qwen3.5-4B is a vision-language checkpoint. LoRA goes on the language model only, and the vision tower stays frozen.

In [ ]:
from unsloth import FastDecisionModel, DecisionTrainer, is_bfloat16_supported

model, tokenizer = FastDecisionModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = LOAD_IN_4BIT,
)

model = FastDecisionModel.get_peft_model(
    model,
    r = LORA_R,
    lora_alpha = LORA_ALPHA,
    lora_dropout = 0,
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
)

print(f"Loaded {MODEL_NAME}, VRAM reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

## 5. Build training items and the calibration holdout

`build_dataset` encodes every row into one prompt (email + both questions + their options). `split_holdout` then keeps 10% of the **train** decisions out of training, using seed 3407. Those rows are the `eval_dataset` during training and the calibration set afterwards, as in the guide. The 400 `test` emails are only used for the before/after numbers.

The holdout depends only on row order and the seed, so it is the same in both notebooks.

In [ ]:
all_train_items, report = FastDecisionModel.build_dataset(train_rows, tokenizer, model)
print(f"train: skipped {report['skipped']} of {report['total']} decisions, truncated {report['truncated']}")
test_items, test_report = FastDecisionModel.build_dataset(test_rows, tokenizer, model)
print(f"test:  skipped {test_report['skipped']} of {test_report['total']} decisions, truncated {test_report['truncated']}")

train_items, calib_items = FastDecisionModel.split_holdout(all_train_items, seed = SEED)
print(f"{len(train_items)} training rows, {len(calib_items)} calibration rows, {len(test_items)} test rows")
print("Calibration row ids:", sorted(item["row"] for item in calib_items)[:10], "...")

## 6. Baseline before training

`FastDecisionModel.evaluate` reports accuracy over all decisions, plus ECE, loss, and `record_accuracy` (both questions right). `score_rows` calls `FastDecisionModel.predict` on each test email, the same call you would use in an app, so you can see accuracy per question. Chance is 12.5% for `category` and 33% for `priority`.

In [ ]:
from tqdm.auto import tqdm

def score_rows(rows, label):
    FastDecisionModel.for_inference(model)
    records = []
    for row in tqdm(rows, desc = label):
        answers = FastDecisionModel.predict(model, tokenizer, json.loads(row["state"]), QUESTIONS)
        cat, pri = answers["category"], answers["priority"]
        records.append({
            "id": row["id"],
            "ambiguous": row["ambiguous"],
            "gold_category": row["category"],
            "pred_category": cat["answer"],
            "p_category": cat["probabilities"][cat["answer"]],
            "gold_priority": row["priority"],
            "pred_priority": pri["answer"],
            "p_priority": pri["probabilities"][str(pri["answer"])],
        })
    frame = pd.DataFrame(records)
    frame["category_ok"] = frame["gold_category"] == frame["pred_category"]
    frame["priority_ok"] = frame["gold_priority"] == frame["pred_priority"]
    return frame

def summarize(frame, label):
    return {
        "run": label,
        "category_acc": frame["category_ok"].mean(),
        "priority_acc": frame["priority_ok"].mean(),
        "both_right": (frame["category_ok"] & frame["priority_ok"]).mean(),
        "category_acc_ambiguous": frame.loc[frame["ambiguous"], "category_ok"].mean(),
    }

base_metrics = FastDecisionModel.evaluate(model, tokenizer, test_items)
print("evaluate():", base_metrics)
base_scored = score_rows(test_rows, "baseline")
pd.DataFrame([summarize(base_scored, "baseline")]).style.format(precision = 3)

## 7. Train

The arguments are the guide's: cosine schedule, 10 warmup steps, weight decay 0.01, and bf16 when the GPU supports it. Checkpoints are off to save disk. With 1,440 training rows and an effective batch of 16, that is 90 steps per epoch, 180 in total. The guide reports Qwen3.5-4B at 76% on typed-decisions after 10 minutes on an L4, and this dataset is smaller. Expect roughly 10–20 minutes on an L4 or 3090 (estimate).

In [ ]:
from transformers import TrainingArguments

torch.cuda.reset_peak_memory_stats()

trainer = DecisionTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = train_items,
    eval_dataset = calib_items,
    args = TrainingArguments(
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        num_train_epochs = EPOCHS,
        learning_rate = LEARNING_RATE,
        lr_scheduler_type = "cosine",
        warmup_steps = 10,
        weight_decay = 0.01,
        bf16 = is_bfloat16_supported(),
        fp16 = not is_bfloat16_supported(),
        eval_strategy = "epoch",
        save_strategy = "no",
        logging_steps = 10,
        output_dir = "outputs",
        report_to = "none",
        seed = SEED,
    ),
)
trainer_stats = trainer.train()

peak_gb = torch.cuda.max_memory_reserved() / 1024**3
print(f"Train time: {trainer_stats.metrics['train_runtime'] / 60:.1f} min, peak VRAM reserved: {peak_gb:.2f} GB")

In [ ]:
import matplotlib.pyplot as plt

history = pd.DataFrame(trainer.state.log_history)
loss = history.dropna(subset = ["loss"]) if "loss" in history else pd.DataFrame()
evals = history.dropna(subset = ["eval_loss"]) if "eval_loss" in history else pd.DataFrame()
plt.figure(figsize = (8, 4))
if len(loss):
    plt.plot(loss["step"], loss["loss"], label = "train loss")
if len(evals):
    plt.plot(evals["step"], evals["eval_loss"], "o-", label = "holdout loss")
plt.xlabel("optimizer step"); plt.ylabel("loss"); plt.legend(); plt.title(MODEL_NAME); plt.show()

## 8. Calibrate

`calibrate` fits temperatures on the held-out calibration rows, so a probability of 0.8 means "right about 80% of the time". The temperatures are stored in the model's config and saved with it.

In [ ]:
calibration = FastDecisionModel.calibrate(model, tokenizer, calib_items)
print(calibration)   # held-out accuracy, calibration error (ece) and loss

## 9. Accuracy after training

In [ ]:
tuned_metrics = FastDecisionModel.evaluate(model, tokenizer, test_items)
tuned_scored = score_rows(test_rows, "fine-tuned")

print("evaluate() before:", {k: round(v, 3) for k, v in base_metrics.items() if isinstance(v, float)})
print("evaluate() after: ", {k: round(v, 3) for k, v in tuned_metrics.items() if isinstance(v, float)})
pd.DataFrame([summarize(base_scored, "baseline"), summarize(tuned_scored, "fine-tuned")]).style.format(precision = 3)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print(classification_report(tuned_scored["gold_category"], tuned_scored["pred_category"], labels = CATEGORIES, digits = 3, zero_division = 0))
print(classification_report(tuned_scored["gold_priority"], tuned_scored["pred_priority"], labels = [0, 1, 2],
                            target_names = ["0 low", "1 normal", "2 urgent"], digits = 3, zero_division = 0))

fig, axes = plt.subplots(1, 2, figsize = (16, 6.5), gridspec_kw = {"width_ratios": [2.2, 1]})
ConfusionMatrixDisplay(
    confusion_matrix(tuned_scored["gold_category"], tuned_scored["pred_category"], labels = CATEGORIES),
    display_labels = CATEGORIES,
).plot(ax = axes[0], xticks_rotation = 45, colorbar = False, cmap = "Blues")
axes[0].set_title("category (test, fine-tuned)")
ConfusionMatrixDisplay(
    confusion_matrix(tuned_scored["gold_priority"], tuned_scored["pred_priority"], labels = [0, 1, 2]),
    display_labels = ["low", "normal", "urgent"],
).plot(ax = axes[1], colorbar = False, cmap = "Greens")
axes[1].set_title("priority (test, fine-tuned)")
plt.tight_layout(); plt.show()

The lowest-confidence and wrong test emails are the ones worth reading. Most of them should be the ambiguous emails that mention two topics.

In [ ]:
mistakes = tuned_scored[~tuned_scored["category_ok"]].sort_values("p_category")
mistakes[["id", "ambiguous", "gold_category", "pred_category", "p_category", "gold_priority", "pred_priority"]].head(15)

## 10. Triage new emails

`predict` takes the email and the questions, which is the same body as a Decision API request. For `category`, `answer` is the option key. For `priority`, it is the level number. `probabilities` covers every option. To act on a choice, threshold its probability, not `confidence`.

In [ ]:
new_emails = [
    {"from": "maria@example.com", "subject": "Charged twice for March",
     "body": "Hi, I was charged twice for my March invoice (INV-2026-4411). Please refund the duplicate today, my card is nearly maxed out."},
    {"from": "ops@corner-bakery.com", "subject": "All cameras offline",
     "body": "Every Lumio Cam 2 in both of our shops went offline after this morning's firmware update. We have no security footage right now."},
    {"from": "tom@example.com", "subject": "Idea",
     "body": "Love the doorbell. Would be great if it could announce visitors on my Sonos speakers. No rush!"},
    {"from": "noreply@seo-rank.net", "subject": "Rank #1 on Google",
     "body": "We can get lumio.example to the first page of Google in 30 days. Reply YES for a free audit."},
    {"from": "jen@example.com", "subject": "Leaving",
     "body": "The app keeps logging me out and I'm tired of it. Please just cancel my Lumio+ Plus plan before it renews on Friday."},
]

FastDecisionModel.for_inference(model)
for email in new_emails:
    answers = FastDecisionModel.predict(model, tokenizer, email, QUESTIONS)
    cat, pri = answers["category"], answers["priority"]
    top = sorted(cat["probabilities"].items(), key = lambda kv: -kv[1])[:3]
    print(f"{email['subject']!r}")
    print(f"  category: {cat['answer']:<18} p={cat['probabilities'][cat['answer']]:.3f}   top-3: {top}")
    print(f"  priority: {pri['answer']} ({PRIORITY_LEVELS[pri['answer']].split(':')[0]})  p={pri['probabilities'][str(pri['answer'])]:.3f}  expected score={pri['score']:.2f}")

## 11. Save the LoRA adapter and the merged model

`save_pretrained` writes the LoRA adapters plus the decision head and its calibration. It is small, and loading it later pulls `unsloth/Qwen3.5-4B` again. `save_pretrained_merged` writes a full 16-bit copy (about 9 GB on disk), which is what the Decision API serves.

In [ ]:
model.save_pretrained(LORA_DIR)
print("Saved adapters + head to", LORA_DIR)

if SAVE_MERGED:
    model.save_pretrained_merged(MERGED_DIR)
    print("Saved merged 16-bit model to", MERGED_DIR)

# from huggingface_hub import notebook_login; notebook_login()
# model.push_to_hub("YOUR_USERNAME/qwen3_5-4b-email-triage")

In [ ]:
!zip -qr {LORA_DIR}.zip {LORA_DIR}
try:
    from google.colab import files
    files.download(f"{LORA_DIR}.zip")
except Exception as exc:
    print(f"Saved {LORA_DIR}.zip (download skipped: {exc})")

### Optional: reload and check

Loading the adapter folder works the same way as loading the base model. Free the training model first so two copies don't share the GPU.

In [ ]:
RELOAD_CHECK = False

if RELOAD_CHECK:
    import gc
    del trainer, model
    gc.collect(); torch.cuda.empty_cache()
    model, tokenizer = FastDecisionModel.from_pretrained(LORA_DIR, load_in_4bit = True)
    FastDecisionModel.for_inference(model)
    print(FastDecisionModel.predict(model, tokenizer, new_emails[0], QUESTIONS)["category"])

## 12. Serve it with the Decision API (Jev / Laya compatible)

This follows Unsloth's [serving guide](https://unsloth.ai/docs/models/decision-laya). Run it on the machine that will host the model, such as your 3090 box, not inside Colab. Copy the merged folder there first, or push it to the Hub.

```bash
curl -fsSL https://unsloth.ai/install.sh | sh          # or: pip install --upgrade unsloth
UNSLOTH_SYSTEMONE_MODEL=/path/to/qwen3_5-4b-email-triage-merged unsloth studio -H 0.0.0.0 -p 8888
```

Then, in Unsloth:

1. Open **Settings → API → Decision API** and turn on **Serve requests**.
2. Copy an API key from **Access tokens**. On localhost you can instead turn on **Keyless API access → Chat and inference**.
3. Requests with `"model": "laya"`, `"default"` or `"jev-latest"` now go to this model. The model needs a GPU and loads on the first request, which can take a few minutes.

The request body is exactly a dataset row's `state` + `questions`:

```bash
curl http://localhost:8888/v1/systemone \
  -H "Authorization: Bearer sk-unsloth-YOUR_KEY" -H "Content-Type: application/json" \
  -d '{"model": "laya", "state": {"subject": "Locked out", "body": "My 2FA codes stopped arriving and I cannot log in."},
       "questions": {"category": {"type": "choice", "instructions": "Which support queue should this customer email be routed to?",
                                  "criteria": {"billing": "...", "account_access": "...", "technical_issue": "..."}}}}'
```

Existing Jev code can use the TypeSafe SDK (`pip install typesafe-sdk`) by setting `TYPESAFE_BASE_URL=http://localhost:8888` and `TYPESAFE_API_KEY=sk-unsloth-...`. Laya's `confidence` uses a different formula from Jev's, so set thresholds on `probabilities`.

The next cell sends one test email to a running server. It does nothing unless you set `DECISION_API_URL`.

In [ ]:
import os, requests

DECISION_API_URL = os.environ.get("DECISION_API_URL", "")   # e.g. "http://localhost:8888"
DECISION_API_KEY = os.environ.get("DECISION_API_KEY", "")   # sk-unsloth-...

body = {"model": "laya", "state": json.loads(test_rows[0]["state"]), "questions": QUESTIONS}
if DECISION_API_URL:
    headers = {"Authorization": f"Bearer {DECISION_API_KEY}"} if DECISION_API_KEY else {}
    response = requests.post(f"{DECISION_API_URL}/v1/systemone", json = body, headers = headers, timeout = 600)
    print(json.dumps(response.json(), indent = 2))
else:
    print("DECISION_API_URL is not set. Request body that would be sent:")
    print(json.dumps(body, indent = 2)[:1200], "...")

## 13. Release GPU memory

In [ ]:
import gc

for name in ("trainer", "model"):
    globals().pop(name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"VRAM reserved now: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

## Notes

- **Synthetic data.** The emails come from templates with random slot values, tones and lengths. Test emails never repeat a train email exactly, but they share templates, so treat these numbers as a demo, not a real-world benchmark. For production, collect a few hundred real emails per queue and keep the same row format.
- **Comparing runs.** [`gemma4_e4b_decision_email_triage.ipynb`](gemma4_e4b_decision_email_triage.ipynb) uses the same files (SHA-256 checked), the same calibration holdout, the same seed and the same effective batch, so its numbers are directly comparable with this notebook's.
- **Less data.** The guide suggests 3–4 epochs on small datasets. This notebook keeps the guide's recommended 2 for the comparison. For a quick smoke test, set `max_steps = 60` in `TrainingArguments`.
- **Out of memory.** Halve `BATCH_SIZE` and double `GRAD_ACCUM`.

### References

- Guide: [Train your own Decision Model with Unsloth](https://unsloth.ai/docs/basics/train-your-own-decision-model-with-unsloth)
- Serving: [Run and Serve Decision Models Locally: Laya + Jev API](https://unsloth.ai/docs/models/decision-laya)
- Dataset format: [`LocalLLaMA/typed-decisions`](https://huggingface.co/datasets/LocalLLaMA/typed-decisions)
- Model: [`unsloth/Qwen3.5-4B`](https://huggingface.co/unsloth/Qwen3.5-4B)